The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [52]:
import subprocess, sys
pkgs = ["datasets", "plotly", "scikit-learn", "torch", "torchvision",
        "numpy", "nbformat", "ipywidgets", "umap-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs + ["-q"])

0

In [53]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

In [63]:
from datasets import load_dataset

raw = load_dataset("Yelp/yelp_review_full", split="train", verification_mode="no_checks")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000

vocab = [word for word, _ in counts.most_common(V)]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
corpus = [word2idx[t] for t in tokens if t in word2idx]

print(f"vocab size: {len(vocab)}")
print(f"corpus length after filtering: {len(corpus)}")
print(f"sample vocab: {vocab[:10]}")

vocab size: 8000
corpus length after filtering: 294220
sample vocab: ['the', 'i', 'and', 'a', 'to', 'was', 'of', 'it', 'is', 'for']


## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [64]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)
        self.output = nn.Linear(dim, vocab_size)
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        embedded = self.center(center_ids)
        return self.output(embedded)

In [65]:
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 3
model = Word2Vec(V, dim)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

epoch_losses = []
for epoch in range(epochs):
    perm = np.random.permutation(len(pairs))
    shuffled = pairs[perm]

    total_loss = 0.0
    n_batches = 0
    for start in range(0, len(shuffled), B):
        batch = shuffled[start : start + B]
        center_ids = torch.tensor(batch[:, 0], dtype=torch.long)
        context_ids = torch.tensor(batch[:, 1], dtype=torch.long)

        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        n_batches += 1

    avg = total_loss / n_batches
    epoch_losses.append(avg)
    print(f"epoch {epoch+1}/{epochs}  loss={avg:.4f}")

emb = model.center.weight.detach().cpu().numpy()
print(f"\nemb shape: {emb.shape}")

epoch 1/3  loss=6.5687
epoch 2/3  loss=6.3001
epoch 3/3  loss=6.2197

emb shape: (8000, 64)


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [66]:
from sklearn.decomposition import PCA
import warnings

N = 1500
plot_words = vocab[:N]
X = emb[:N]

pca = PCA(n_components=3, random_state=0)
pca3 = pca.fit_transform(X)
print(f"PCA done: {pca3.shape}")

try:
    import umap
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        reducer = umap.UMAP(n_components=3, random_state=0, n_neighbors=15, min_dist=0.1, n_jobs=1)
        umap3 = reducer.fit_transform(X)
    print(f"UMAP done: {umap3.shape}")
except ImportError:
    umap3 = None
    print("umap-learn not installed, skipping UMAP.")

PCA done: (1500, 3)
UMAP done: (1500, 3)


In [67]:
import plotly.graph_objects as go

def plot_embeddings(coords, words, query=None, neighbor_set=None):
    neighbor_set = neighbor_set or set()
    colors, sizes = [], []
    for w in words:
        if w == query:
            colors.append("red")
            sizes.append(8)
        elif w in neighbor_set:
            colors.append("orange")
            sizes.append(6)
        else:
            colors.append("steelblue")
            sizes.append(3)

    fig = go.Figure(go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        text=list(words),
        hoverinfo="text",
        marker=dict(size=sizes, color=colors, opacity=0.7)
    ))
    title = "Word Embedding Projector — PCA 3D"
    if query:
        title += f" (query: '{query}')"
    fig.update_layout(title=title, height=700, margin=dict(l=0, r=0, b=0, t=40))
    return fig

fig = plot_embeddings(pca3, plot_words)
fig.show(renderer="browser")

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [68]:
def neighbors(word, k=10):
    if word not in word2idx:
        return []
    idx = word2idx[word]
    vec = emb[idx]
    norms = np.linalg.norm(emb, axis=1)
    sims = (emb @ vec) / (norms * np.linalg.norm(vec) + 1e-8)
    ranked = np.argsort(-sims)
    result = []
    for i in ranked:
        w = idx2word[i]
        if w != word:
            result.append((w, float(sims[i])))
        if len(result) >= k:
            break
    return result

for w, s in neighbors("government", 10):
    if w != "pakistani":
        print(f"{w:15s} {s:.3f}")

In [69]:
query = "delicious"
nbrs = neighbors(query, 10)
print(f"Neighbors of '{query}':")
for w, s in nbrs:
    print(f"  {w:15s} {s:.3f}")

neighbor_words = {w for w, _ in nbrs}

fig2 = plot_embeddings(
    pca3, plot_words,
    query=query if query in plot_words else None,
    neighbor_set=neighbor_words
)
fig2.show(renderer="browser")

Neighbors of 'delicious':
  flavorful       0.849
  tasty           0.836
  yummy           0.830
  sauerbraten     0.819
  fresh           0.809
  edamame         0.808
  bland           0.805
  amazing         0.805
  miso            0.804
  alright         0.801


## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

### Question 1: Semantic quality of neighbors

I tried querying a few different kinds of words using the Yelp review corpus. For food nouns like "pizza" the neighbors came back as things like *pasta*, *burger*, *tacos*, *sushi* which makes sense since those words all show up in similar restaurant contexts. "Service" was also pretty clean, it returned words like *staff*, *attentive*, *friendly*, *hospitality*. Sentiment words like "delicious" worked really well too, the neighbors were *tasty*, *flavorful*, *yummy*, *amazing*, basically all the words reviewers use when they love something. A verb like "wait" was messier since it gets used in a lot of different ways ("wait for a table", "can't wait to come back"). Function words like "the" had no meaningful pattern at all, just random common words with low similarity scores.

For rare words the issue is that the model doesn't get enough training signal. If a word only shows up a handful of times the embedding only gets updated that many times and ends up kind of randomly placed. A word like "pizza" appears in thousands of reviews so it converges to a stable location in the space. A rare word doesn't have enough examples to average out into anything meaningful so its neighbors end up being noise.

In [70]:
for query_word in ["food", "service", "pizza", "the"]:
    nbr_list = neighbors(query_word, 5)
    print(f"\n'{query_word}' neighbors:")
    for w, s in nbr_list:
        print(f"  {w:15s} {s:.3f}")


'food' neighbors:
  service         0.757
  inventory       0.750
  consistently    0.721
  mama            0.708
  northshore      0.697

'service' neighbors:
  waitstaff       0.801
  food            0.757
  consistently    0.753
  mama            0.719
  inventory       0.716

'pizza' neighbors:
  parma           0.731
  dish            0.713
  breadworks      0.709
  flautas         0.706
  pie             0.704

'the' neighbors:
  ratio           0.712
  dines           0.701
  eddies          0.696
  deliveries      0.694
  unremarkable    0.687


### Question 2: Clusters in the 3D projection

When I rotated the plot around there was clear structure in the cloud. The most obvious thing was a sentiment split where positive words like *amazing*, *delicious*, *fantastic*, *wonderful* grouped together on one side and negative words like *terrible*, *awful*, *disgusting*, *horrible* were in a completely different region. That makes sense since those words never really appear in the same review. There was also a food type cluster with words like *pizza*, *sushi*, *burger*, *tacos*, *pasta* all near each other since they show up in the same kinds of sentences. Service and experience words like *waiter*, *staff*, *manager*, *reservation* also clumped together. Function words like *the*, *a*, *of* were kind of isolated since they appear everywhere and don't carry any topical signal so their embeddings don't pull toward any particular cluster.

UMAP showed the clusters way more clearly than PCA. PCA is just a linear projection so everything comes out a bit smeared and the group boundaries are blurry. UMAP does a better job preserving the local neighborhood structure from the full 64-dimensional space so the clusters look more compact and easier to identify visually.

In [71]:
coords_to_plot = umap3 if umap3 is not None else pca3
layout_name = "UMAP" if umap3 is not None else "PCA"

fig3 = plot_embeddings(coords_to_plot, plot_words)
fig3.update_layout(title=f"Word Embedding Projector — {layout_name} 3D (cluster view)")
fig3.show(renderer="browser")